## <div align='center'>NASA Boeing 777-200ER (N577NA) range Estimates for Antarctica and the Arctic</div>

This Jupyter notebook estimates ranges over Antarctica for the NASA Boeing 777 (B777-200ER) a.k.a. 'the triple seven' airborne science laboratory.

### 1) Load required Python modules, processing options, parameters and necessary maps, and define classes/functions:

In [ ]:
import geopandas as gpd
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import colors
from osgeo import gdal
from pyproj import Geod, Transformer
from scipy.interpolate import RegularGridInterpolator, PchipInterpolator

gdal.UseExceptions()

takeoff_penalty	    = 15 / 60 #  time it takes to get on track following takeoff, hr
landing_penalty	    = 15 / 60 # time it takes to get lined up for landing, hr
time_inc		    = 5 / 60 # time increment, hr
decim_bm            = 50 # decimation ratio for BedMachine grid (50 equivalent 50 * 500 m = 25 km resolution)
decim_ibcao         = 50 # decimation ratio for IBCAO grid (50 equivalent 50 * 400 m = 20 km resolution)
rad_circle          = np.deg2rad(np.arange(360 + 1)) # radians for full circle, 0 to 360 deg
num_circle          = len(rad_circle)
geoid               = Geod(ellps='WGS84') # geoid instance, here we're using WGS84 ellipsoid
time_max_pituffik   = 8 # maximum flight time from Pituffik, hr

# BedMachine Antarctica v4 file name, https://nsidc.org/data/nsidc-0756/versions/4
path_bm = '/Users/jamacgre/OneDrive - NASA/research/data/antarctica/BedMachine/'
file_bm = 'NSIDC-0756_BedMachineAntarctica_19700101-20191001_V04.1.nc'

path_moa = '/Users/jamacgre/OneDrive - NASA/research/data/moa/new/'
# MOA 2008-9 mosaic/coastline/grounding-line/islands in EPSG:3031 already, https://nsidc.org/data/nsidc-0593/versions/2 and https://www.usap-dc.org/view/dataset/609593
file_moa            = 'moa750_2009_hp1_v1.1.tif'
file_coastline      = 'moa_2009_coastline_v1.1.shp'
file_grounding_line = 'moa_2009_groundingline_v1.1.shp'
file_islands        = 'moa_2009_islands_v1.1.shp'

# IBCAO
path_ibcao = '/Users/jamacgre/Library/CloudStorage/OneDrive-NASA/data/777_Arctic/'
file_ibcao = 'ibcao_v5_2_2026_ice_400m.tiff'

# aircraft class, with 777 defaults
class Aircraft():
    def __init__(self):
        self.Name              = 'NASA B777-200ER' # aircraft name
        self.ShortName         = '777'             # aircraft short name
        self.MTOW              = 650e3             # maximum take-off weight, lb
        self.OEW               = 321e3             # operating empty weight incl. crew, lb
        self.MissionLoad       = 45e3              # load for mission operations incl. QNCs + instruments, lb
        self.FuelTotal         = 308e3             # fuel total, lb
        self.FuelTotalOp       = 284e3             # operational fuel total (assuming tanks are not completely filed), lb
        self.FuelReserveTime   = 30                # minimum fuel reserve, lb
        self.SpeedCruise       = 460               # high-altitude cruise speed, kt
        self.SpeedLo           = 275               # low-alititude survey speed, kt
        self.SpeedLoPenalty    = 2                 # fuel burn penalty ratio for jet low-flying compared to cruise, dimensionless
        self.ETOPS             = 330               # maximum ETOPS for aircraft type (B777-200ER), min
        self.SpeedETOPS        = 400               # one-engine ETOPS airspeed, kt
        self.SpeedETOPSPenalty = 1.5               # one-engine ETOPS fuel burn "penalty" ratio, dimensionless
        
# survey class, empty defaults
class Survey():
    def __init__(self):
        self.DistTransit       = np.nan # transit distance between base and target, m
        self.TimeTransit       = np.nan # transit time, hr
        self.FuelTransitOut    = np.nan # fuel required for transit out to target, lb
        self.FuelTransitReturn = np.nan # fuel required for return to base, lb
        self.FuelTransit       = np.nan # total fuel required for out-and-back transit, lb
        self.RangeSurvey       = np.nan # available survey range, m
        self.TimeSurvey        = np.nan # available survey time, hr
        self.XRangeCircle      = np.empty(num_circle, dtype=float) # projected x of survey range circle, m
        self.YRangeCircle      = np.empty(num_circle, dtype=float) # projected y of survey range circle, m

# survey_range is a function to calculate survey range/time based on aircraft parameters (ac), and EITHER
# already known distance (dist) 
# OR
# base (lat1/lon1) and target (lat2/lon/x2/y2), also need geoid instance (geoid) and vector of circle radians (rad_circle)
def survey_range(ac, **kwargs):
    
    survey = Survey()
    
    if 'dist' not in kwargs:
        _, _, survey.DistTransit = kwargs.get('geoid').inv(kwargs.get('lon1'), 
                                                           kwargs.get('lat1'),
                                                           kwargs.get('lon2'),
                                                           kwargs.get('lat2')) # great-circle transit distance between base and target, m
    else:
        survey.DistTransit = kwargs.get('dist')

    if 'time_max' in kwargs:
        survey.TimeMax = kwargs.get('time_max')
    else:
        survey.TimeMax = np.inf

    survey.TimeTransit = survey.DistTransit / ac.SpeedCruise # transit time, hr
    
    # stop immediately if transit time is too long to survey usefully
    if (2 * survey.TimeTransit) > survey.TimeMax:
        return survey

    # initialize distance, fuel and load
    dist_curr = 0
    time_curr = 0
    fuel_curr = ac.FuelTotalOp
    load_curr = ac.LoadStart
    
    # add in takeoff penalty
    for takeoff in range(np.round(takeoff_penalty / time_inc).astype(int)):
        fuel_burn_curr = ac.BurnRateHiInterp(load_curr) * time_inc # current fuel burn in lbs = rate (lb/hr) * hr
        fuel_curr      -= fuel_burn_curr # fuel weight available - weight burned = weight remaining
        load_curr      -= fuel_burn_curr / ac.LoadMax # previous load - load just spent (which was all fuel)
        time_curr      += time_inc # increment time by time increment, hr
    
    # keep going until you reach target
    while (dist_curr < survey.DistTransit) and (fuel_curr > ac.FuelReserveWt) and (time_curr < survey.TimeMax):
        dist_curr      += time_inc * ac.SpeedCruise # hr * m/hr = m
        fuel_burn_curr = ac.BurnRateHiInterp(load_curr) * time_inc
        fuel_curr      -= fuel_burn_curr
        load_curr      -= fuel_burn_curr / ac.LoadMax
        time_curr      += time_inc # increment time by time increment, hr
    
    survey.FuelTransitOut = ac.FuelTotalOp - fuel_curr # fuel needed to transit out to target, lb
    
    # now reverse, starting from base on return, first adding in landing penalty, but base this on ETOPS
    dist_curr = 0
    fuel_curr = ac.FuelReserveWt
    load_curr = ac.LoadEnd
    
    for landing in range(np.round(landing_penalty / time_inc).astype(int)):
        fuel_burn_curr = ac.BurnRateHiInterp(load_curr) * time_inc
        fuel_curr      += fuel_burn_curr
        load_curr      += fuel_burn_curr / ac.LoadMax
        time_curr      += time_inc
    
    while dist_curr < survey.DistTransit:
        dist_curr      += time_inc * ac.SpeedCruise
        fuel_burn_curr = ac.BurnRateHiInterp(load_curr) * time_inc
        fuel_curr      += fuel_burn_curr
        load_curr      += fuel_burn_curr / ac.LoadMax
        time_curr      += time_inc

    survey.FuelTransitReturn = fuel_curr - ac.FuelReserveWt # fuel needed to get back from target under ETOPS, lb
    survey.FuelTransit       = survey.FuelTransitOut + survey.FuelTransitReturn # total transit fuel, lb
    
    # stop if too far away to survey or not enough time
    if (survey.FuelTransit >= (ac.FuelTotalOp - ac.FuelReserveWt) or (time_curr >= survey.TimeMax)):
        return survey
    
    # calculate distance & time at survey altitude/speed on-station
    dist_curr = 0
    fuel_curr = ac.FuelTotalOp - survey.FuelTransitOut
    load_curr = ac.LoadStart - (survey.FuelTransitOut / ac.LoadMax)
    
    while ((fuel_curr > (survey.FuelTransitReturn + fuel_ETOPS_interp((survey.DistTransit, fuel_curr)))) and (time_curr <= survey.TimeMax)): # keep surveying until it's time to turn around
        dist_curr      += time_inc * ac.SpeedLo # now use low-altitude survey speed
        fuel_burn_curr = ac.BurnRateLoInterp(load_curr) * time_inc
        fuel_curr      -= fuel_burn_curr
        load_curr      -= fuel_burn_curr / ac.LoadMax
        time_curr      += time_inc
    
    survey.RangeSurvey = dist_curr # full range survey
    survey.TimeSurvey  = dist_curr / ac.SpeedLo # time for full range survey
    
    if 'dist' not in kwargs:
        survey.XRangeCircle, survey.YRangeCircle = kwargs.get('x2') + (np.cos(kwargs.get('rad_circle')) * (survey.RangeSurvey / 2)), \
                                                   kwargs.get('y2') + (np.sin(kwargs.get('rad_circle')) * (survey.RangeSurvey / 2)) # out-and-back range-to-target circle, projected x/y, m
    
    return survey

# fuel needed for ETOPS return
def fuel_ETOPS(ac, dist_transit, fuel_curr, load_curr, time_inc):
    fuel_start  = fuel_curr
    dist_return = dist_transit
    while dist_return > 0:
        dist_return    -= time_inc * ac.SpeedETOPS
        fuel_burn_curr = (ac.BurnRateHiInterp(load_curr) / ac.SpeedETOPSPenalty) * time_inc
        fuel_curr      -= fuel_burn_curr
        load_curr      -= fuel_burn_curr / ac.LoadMax
    fuel_return = fuel_start - fuel_curr # fuel needed for return is what we started with minus what's left
    return fuel_return

### 2) Further work on aircraft performance parameters

### 3a) Load BedMachine mask, MOA and associated products (Antarctica)

In [ ]:
bm = xr.open_dataset(path_bm + file_bm)

# extract x/y
x_bm         = bm.x.values.astype(np.float32)
y_bm         = bm.y.values.astype(np.float32)
xx_bm, yy_bm = np.meshgrid(x_bm, 
                           y_bm)

# decimate x/y and get lat/lon
x_decim, y_decim     = x_bm[::decim_bm], 
y_bm[::decim_bm]
xx_decim, yy_decim   = np.meshgrid(x_decim, 
                                   y_decim)
lat_decim, lon_decim = Transformer.from_crs('EPSG:3031', 
                                            'EPSG:4326').transform(xx_decim, 
                                                                   yy_decim)

# simplify mask
mask_bm              = bm['mask'].values
mask_bm[mask_bm > 0] = 1
mask_bm_decim        = mask_bm[::decim_bm, 
                               ::decim_bm].astype(bool)

bm.close()

# load/extract MOA mosaic and get x/y limits
moa_inst = gdal.Open(path_moa + file_moa)
moa      = moa_inst.GetRasterBand(1).ReadAsArray()
moa_gt   = moa_inst.GetGeoTransform()
x_min_moa, x_max_moa, y_min_moa, y_max_moa = moa_gt[0], moa_gt[0] + (moa_gt[1] * moa_inst.RasterXSize), moa_gt[3], moa_gt[3] + (moa_gt[5] * moa_inst.RasterYSize)
moa_inst.Close()

# load MOA 2008-9 coastline, grounding line and islands
moa_cl = gpd.read_file(path_moa + file_coastline)
moa_gl = gpd.read_file(path_moa + file_grounding_line)
moa_il = gpd.read_file(path_moa + file_islands)

### 2b) OR Load IBCAO (Arctic)

In [ ]:
# IBCAO in EPSG:3996

ibcao_inst = gdal.Open(path_ibcao + file_ibcao)
ibcao      = ibcao_inst.GetRasterBand(1).ReadAsArray()
ibcao_gt   = ibcao_inst.GetGeoTransform()
x_min_ibcao, x_max_ibcao, y_min_ibcao, y_max_ibcao = ibcao_gt[0], ibcao_gt[0] + (ibcao_gt[1] * ibcao_inst.RasterXSize), ibcao_gt[3], ibcao_gt[3] + (ibcao_gt[5] * ibcao_inst.RasterYSize)
ibcao_inst.Close()

x_ibcao             = np.arange(x_min_ibcao, 
                                (x_max_ibcao + ibcao_gt[1]), 
                                ibcao_gt[1])
y_ibcao             = np.arange(y_min_ibcao,
                                (y_max_ibcao + ibcao_gt[5]), 
                                ibcao_gt[5])
xx_ibcao, yy_ibcao = np.meshgrid(x_ibcao, 
                                 y_ibcao)

# decimate x/y and get lat/lon
x_decim, y_decim     = x_ibcao[::decim_ibcao], y_ibcao[::decim_ibcao]
xx_decim, yy_decim   = np.meshgrid(x_decim, 
                                   y_decim)
lat_decim, lon_decim = Transformer.from_crs('EPSG:3996', 
                                            'EPSG:4326').transform(xx_decim, 
                                                                   yy_decim)

In [ ]:
# create return variable 'ac' as an Aircraft class with fields populated
ac = Aircraft()

# 777 specific stats from https://community.infiniteflight.com/t/your-guide-to-fuel-burn-and-cruising-altitudes-in-the-new-777-family-ignore-777f/488605
load_range_ref     = np.arange(0.1, 
                               (1.0 + 0.1), 
                               0.1)
fuel_burn_per_load = np.vstack((load_range_ref,
                                (1e3 * np.array((8.5, 9.0, 10.4, 10.9, 11.8, 12.4, 13.2, 14.1, 14.9, 15.8))),
                                (1e3 * np.array((10.0, 10.5, 11.7, 12.3, 12.2, 12.9, 13.9, 14.6, 15.7, 16.7))))) # row 1: percentage load; row 2: lowest burn rate in lbs/hr; row 3: lbs/hr higest of burn rates at that load
fuel_burn_per_load = np.vstack((fuel_burn_per_load, 
                                np.mean(fuel_burn_per_load[1:3, :], 
                                        axis=0))) # mean of high and low

# update parameters and calculate more as needed
ac.SpeedCruise *= 1.852e3 # convert from kt to m/hr
ac.SpeedLo     *= 1.852e3
ac.SpeedETOPS  *= 1.852e3
ac.WtStart     = ac.OEW + ac.MissionLoad + ac.FuelTotalOp # total starting weight, lb
ac.WtSpare     = ac.MTOW - ac.WtStart # spare (unused) weight, lb
if ac.WtSpare < 0: # need some spare weight!
    print(ac.Name + ' too heavy!!! (starting weight > MTOW)')
ac.LoadMax          = ac.MTOW - ac.OEW # max load, lb
ac.BurnRateHiRange  = fuel_burn_per_load[3, :] # using mean of internet values from above
ac.BurnRateLoRange  = ac.SpeedLoPenalty * ac.BurnRateHiRange # penalty for low-flying with jet, based on G-V experience
ac.BurnRateHiInterp = PchipInterpolator(load_range_ref, 
                                        ac.BurnRateHiRange, 
                                        extrapolate=True)
ac.BurnRateLoInterp = PchipInterpolator(load_range_ref, 
                                        ac.BurnRateLoRange, 
                                        extrapolate=True)
ac.FuelReserveWt = (ac.FuelReserveTime / 60) * ac.BurnRateHiInterp((ac.OEW + ac.MissionLoad + (0.2 * ac.FuelTotalOp)) / ac.LoadMax) # assume reserve only needed at lower load, lb
ac.WtEnd         = ac.OEW + ac.MissionLoad + ac.FuelReserveWt # weight at end assuming we still have reserve, lb
ac.LoadStart     = (ac.WtStart - ac.OEW) / ac.LoadMax # load ratio at start, dimensionless
ac.LoadEnd       = (ac.WtEnd  - ac.OEW) / ac.LoadMax # load ratio at end, dimensionless
ac.RangeETOPS    = (ac.ETOPS / 60) * ac.SpeedETOPS # maximum range based on ETOPS rules, m

# generate 2-D interpolant for distance/fuel and ETOPS 
dist_inc          = 50e3 # distance increment to test, m
dist_ETOPS_test   = np.arange(dist_inc, (ac.RangeETOPS + dist_inc), dist_inc)
fuel_ETOPS_test   = np.linspace(ac.FuelReserveWt, ac.FuelTotalOp, len(dist_ETOPS_test))
load_ETOPS_test   = (ac.OEW + ac.MissionLoad + ac.FuelReserveWt + fuel_ETOPS_test) / ac.LoadMax
fuel_ETOPS_interp = RegularGridInterpolator((dist_ETOPS_test, fuel_ETOPS_test), 
                                            np.array([[fuel_ETOPS(ac, dist, fuel, load, time_inc) 
                                                       for fuel, load in zip(fuel_ETOPS_test, load_ETOPS_test)] 
                                                       for dist in dist_ETOPS_test]), 
                                            bounds_error=False, 
                                            fill_value=None) # 2-D linear interpolant for ETOPS fuel calculation

### 4a) Load bases of operation (Antarctica)

In [ ]:
file_base = 'bases_of_operation_antarctica.csv'
base_df = pd.read_csv(file_base)

# not needed here but is good practice: wrap longitudes to ±180° for exporting geographic coordinates 
# 0° to 360° is not supported for GeoPackage (GPKG) format    
base_df['Longitude'] = np.mod(base_df['Longitude'] - 180.0, 360.0) - 180.0

base_gdf = gpd.GeoDataFrame(base_df,
                            geometry=gpd.points_from_xy(base_df['Longitude'], 
                                                        base_df['Latitude']))
# set up the coordinate system for GeoDataPackage
base_gdf = base_gdf.set_crs('EPSG:4326')
# save GeoDataPackage if desired
base_gdf.to_file(file_base.replace('.csv', 
                                   '.gpkg'), 
                 driver='GPKG')

num_base = len(base_gdf)

num_divert = 0

base_gdf

,Name,Country,CountryNice,ICAO,Latitude,Longitude,Color,geometry
0,Ushuaia,ARG,Argentina,SAWH,-54.84,-68.30,r,POINT (-68.3 -54.84)
1,Hobart,AUS,Australia,YMHB,-42.84,147.51,b,POINT (147.51 -42.84)
2,Cape Town,ZAF,South Africa,FACT,-33.97,18.60,y,POINT (18.6 -33.97)


### 4b) Load bases of operation (Arctic)

In [ ]:
# Svalbard can only be an ETOPS divert, not a basing location

file_base = 'bases_of_operation_arctic.csv'
base_df = pd.read_csv(file_base)

# not needed here but is good practice: wrap longitudes to ±180° for exporting geographic coordinates 
# 0° to 360° is not supported for GeoPackage (GPKG) format    
base_df['Longitude'] = np.mod(base_df['Longitude'] - 180.0, 360.0) - 180.0

base_gdf = gpd.GeoDataFrame(base_df,
                            geometry=gpd.points_from_xy(base_df['Longitude'], 
                                                        base_df['Latitude']))
# set up the coordinate system for GeoDataPackage
base_gdf = base_gdf.set_crs('EPSG:4326')
# save GeoDataPackage if desired
base_gdf.to_file(file_base.replace('.csv', 
                                   '.gpkg'), 
                 driver='GPKG')

divert_gdf = base_gdf.copy()
divert_gdf = divert_gdf[divert_gdf['Divert'] == 1] # only keep divert bases
num_divert = len(divert_gdf)

base_gdf = base_gdf[base_gdf['Divert'] == 0] # remove divert bases from basing locations
base_gdf = base_gdf.reset_index()

num_base = len(base_gdf)

base_gdf
# divert_gdf

### 5) Load example targets (Antarctica, no longer used)

In [ ]:
file_target = '2019_Antarctic_targets.kml'

target_gdf = gpd.read_file(file_target, 
                           columns=['Name', 'geometry'])
target_gdf.geometry = target_gdf.geometry.force_2d() # remove z

for target_to_remove in ['88S traverse center', 'Rennick GZ', 'Cook GZ', 'Ninnis GZ', 'Français GZ', 'Underwood GZ', 'Conger GZ', 'Dibble GZ']:
    target_gdf = target_gdf.drop(target_gdf[target_gdf['Name'] == target_to_remove].index)
target_gdf.reset_index(inplace=True)

num_target = len(target_gdf)

target_gdf_proj = target_gdf.copy()
target_gdf_proj = target_gdf.to_crs(epsg=3031)

target_gdf
# target_gdf_proj

In [ ]:
# import existing atmospheric science surveys

# ACE-SPACE amazing name
path_ace = '/Users/jamacgre/OneDrive - NASA/research/funding/777/atmos_tracks/ACE-SPACE/'
file_ace = 'cruise-track-5min-legs0-4.csv'

ace_gpd_raw = gpd.read_file(path_ace + file_ace)

ace_gpd = gpd.GeoDataFrame(ace_gpd_raw,
                           geometry=gpd.points_from_xy(ace_gpd_raw.longitude,
                                                       ace_gpd_raw.latitude),
                                                       crs="EPSG:4326").to_crs("EPSG:3031")

In [ ]:
# SOCRATES
path_socrates = '/Users/jamacgre/OneDrive - NASA/research/funding/777/atmos_tracks/SOCRATES/'
dir_socrates = sorted(glob.glob(path_socrates + '*.kml'))
socrates_gpd = [None] * len(dir_socrates)
for i, file_socrates in enumerate(dir_socrates):
    socrates_raw = gpd.read_file(file_socrates,
                                 layer='rf01')
    # socrates_gpd[i] = gpd.GeoDataFrame(socrates_raw
    #                                    geometry=gpd.points_from_xy(socrates_raw.geometry[0].x, 
    #                                                                socrates_raw.geometry[0].y), 
    #                                                                crs="EPSG:4326").to_crs("EPSG:3031")
    # socrates_gpd['Name'] = 'SOCRATES cruise ' + str(i+1)
    # if i == 0:
    #     socrates_gpd_all = socrates_gpd
    # else:
    #     socrates_gpd_all = pd.concat([socrates_gpd_all, socrates_gpd], ignore_index=True)


In [ ]:
gpd.read_file(file_socrates, layer='rf01').geometry[0].xy

### 6) Calculate range-at-target survey distance and time (no longer used, Antarctica only)

In [ ]:
survey = [None] * num_base

# loop through each base then each target
for base in range(num_base):
    
    survey[base] = [None] * num_target
    
    for target in range(num_target):
        
        survey[base][target] = survey_range(ac, 
                                            lat1=base_gdf.geometry[base].y, 
                                            lon1=base_gdf.geometry[base].x, 
                                            lat2=target_gdf.geometry[target].y, 
                                            lon2=target_gdf.geometry[target].x, 
                                            x2=target_gdf_proj.geometry[target].x, 
                                            y2=target_gdf_proj.geometry[target].y, 
                                            geoid=geoid, 
                                            rad_circle=rad_circle)

### 7a) Calculate Antarctic-wide gridded range-at-target survey distance and time

In [21]:
decim_shape = (mask_bm_decim.shape[0], mask_bm_decim.shape[1], num_base)
dist_grd, range_grd, time_grd, time_transit_grd, dist_ETOPS = np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape[0:2], True)

# great-circle distance between each base and decimated BedMachine grid
for base in range(num_base):
    _, _, dist_grd[:, :, base] = geoid.inv(np.full(decim_shape[0:2], base_gdf.geometry[base].x), 
                                           np.full(decim_shape[0:2], base_gdf.geometry[base].y), 
                                           lon_decim,
                                           lat_decim)
dist_grd_min = np.nanmin(dist_grd,
                         axis=2)
dist_ETOPS[dist_grd_min > ac.RangeETOPS] = False # ETOPS distance evaluation for easy contouring later on

# indices where ice is present so we'll test them
for i in range(decim_shape[0]):
    print(i)
    for j in range(decim_shape[1]):    
        for base in range(num_base):
            survey_curr = survey_range(ac, 
                                       dist=dist_grd[i, j, base])        
        
            range_grd[i, j, base]        = survey_curr.RangeSurvey
            time_grd[i, j, base]         = survey_curr.TimeSurvey
            time_transit_grd[i, j, base] = survey_curr.TimeTransit

range_grd_max        = np.nanmax(range_grd,
                                 axis=2)
time_grd_max         = np.nanmax(time_grd,
                                 axis=2)
time_transit_grd_min = np.nanmin(time_transit_grd,
                                 axis=2)

range_grd_max[dist_ETOPS == False]        = np.nan
time_grd_max[dist_ETOPS == False]         = np.nan
time_transit_grd_min[dist_ETOPS == False] = np.nan

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266


In [ ]:
# OLD OLD OLD
# 
# # indices where ice is present so we'll test them
# ind_mask = np.argwhere(mask_bm_decim)

# for pt in range(len(ind_mask)):
    
#     if pt % 1e3 == 0:
#         print(str(1e2 * (pt / len(ind_mask))) + '%...')
    
#     for base in range(num_base):
        
#         survey_curr = survey_range(ac, 
#                                    dist=dist_grd[ind_mask[pt][0], ind_mask[pt][1], base])        
        
#         range_grd[ind_mask[pt][0], ind_mask[pt][1], base]        = survey_curr.RangeSurvey
#         time_grd[ind_mask[pt][0], ind_mask[pt][1], base]         = survey_curr.TimeSurvey
#         time_transit_grd[ind_mask[pt][0], ind_mask[pt][1], base] = survey_curr.TimeTransit


In [ ]:
import rasterio
transform = rasterio.transform.from_origin(x_decim[0], 
                                           y_decim[0], 
                                           (500 * decim_bm), 
                                           (500 * decim_bm))
with rasterio.open(
    '/Users/jamacgre/OneDrive - NASA/data/777_Antarctic/777_survey_range_Antarctic.tif',           # Output filename
    'w',                    # Write mode
    driver='GTiff',         # Specify GeoTIFF driver
    height=range_grd_max.shape[1],   # Array height (rows)
    width=range_grd_max.shape[0],    # Array width (columns)
    count=1,                # Number of bands (1 for 2D array)
    dtype=range_grd_max.dtype,       # Data type (e.g., float32, uint16)
    crs='EPSG:3031',                # Coordinate Reference System
    transform=transform     # Affine transformation
) as dst:
    # Write the array to band 1
    dst.write(range_grd_max / 1e3, 1)

### 7b) Calculate Arctic-wide gridded range-at-target survey distance and time

In [ ]:
decim_shape = (xx_decim.shape[0], xx_decim.shape[1], num_base)
dist_grd, range_grd, time_grd, time_transit_grd, dist_ETOPS = np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape, np.nan), \
                                                              np.full(decim_shape[0:2], True)

# great-circle distance between each base and decimated IBCAO grid
for base in range(num_base):
    _, _, dist_grd[:, :, base] = geoid.inv(np.full(decim_shape[0:2], base_gdf.geometry[base].x), 
                                           np.full(decim_shape[0:2], base_gdf.geometry[base].y), 
                                           lon_decim,
                                           lat_decim)

dist_grd_min = np.nanmin(dist_grd, 
                         axis=2)
dist_ETOPS[dist_grd_min > ac.RangeETOPS] = False # ETOPS distance evaluation for easy contouring later on

# indices where ice is present so we'll test them
for i in range(decim_shape[0]):
    print(i)
    for j in range(decim_shape[1]):    
        for base in range(num_base):
            if base_gdf['Name'][base] == 'Pituffik':
                survey_curr = survey_range(ac, 
                                           dist=dist_grd[i, j, base],
                                           time_max=time_max_pituffik)
            else:
                survey_curr = survey_range(ac, 
                                           dist=dist_grd[i, j, base])      
        
            range_grd[i, j, base]        = survey_curr.RangeSurvey
            time_grd[i, j, base]         = survey_curr.TimeSurvey
            time_transit_grd[i, j, base] = survey_curr.TimeTransit

range_grd_max = np.nanmax(range_grd, 
                          axis=2)
time_grd_max = np.nanmax(time_grd, 
                         axis=2)
time_transit_grd_min = np.nanmin(time_transit_grd, 
                                 axis=2)

range_grd_max[dist_ETOPS == False] = np.nan
time_grd_max[dist_ETOPS == False] = np.nan
time_transit_grd_min[dist_ETOPS == False] = np.nan

russia_inst = gdal.Open(path_ibcao + 'Russia_mask_IBCAO.tif')
russia      = russia_inst.GetRasterBand(1).ReadAsArray().astype(bool)
russia_inst.Close()

range_grd_russia = np.copy(range_grd_max)
range_grd_russia[~russia] = np.nan

plt.figure()
plt.imshow(range_grd_russia)

In [ ]:
import rasterio
transform = rasterio.transform.from_origin(x_decim[0], 
                                           y_ibcao[0], 
                                           (ibcao_gt[1] * decim_ibcao), 
                                           (-ibcao_gt[5] * decim_ibcao))
with rasterio.open(
    '/Users/jamacgre/OneDrive - NASA/data/777_Antarctic/777_survey_range_Arctic.tif', # Output filename
    'w',                    # Write mode
    driver='GTiff',         # Specify GeoTIFF driver
    height=range_grd_max.shape[1],   # Array height (rows)
    width=range_grd_max.shape[0],    # Array width (columns)
    count=1,                # Number of bands (1 for 2D array)
    dtype=range_grd_max.dtype,       # Data type (e.g., float32, uint16)
    crs='EPSG:3996',                # Coordinate Reference System
    transform=transform     # Affine transformation
) as dst:
    # Write the array to band 1
    dst.write(range_grd_russia / 1e3, 1)

### 8a) Plot range-to-target grids (Antarctica)

In [ ]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size']   = 6
plt.rcParams['font.weight'] = 'bold'

num_color = 6
range_set = np.linspace(1.5e3, 4.5e3, 1 + num_color)
cmap = plt.colormaps['cool'].resampled(num_color)
fig = plt.figure(figsize=(4, 4), dpi=300, facecolor='w', layout='constrained')
ax = plt.subplot(1, 1, 1)
plt.imshow(moa[::10, ::10], 
           extent=(x_min_moa, x_max_moa, y_max_moa, y_min_moa), 
           aspect='equal', 
           cmap='gray')
ims, pc = [None] * num_base, [None] * num_base
for base in range(num_base):
    alpha_curr = np.full(range_grd.shape[0:2], 0.7)
    alpha_curr[~mask_bm_decim] = 0.5
    alpha_curr[dist_grd_min > ac.RangeETOPS] = 0
    ims[base] = plt.imshow(range_grd[:, :, base] / 1e3, 
                           extent=(np.min(x_bm), np.max(x_bm), np.min(y_bm), np.max(y_bm)), 
                           aspect='equal', 
                           cmap=colors.ListedColormap(cmap(range(num_color))), 
                           vmin=range_set[0],
                           vmax=range_set[-1],
                           alpha=alpha_curr)
    plt.contour(x_decim, y_decim, dist_ETOPS[:, :, base], levels=[0, 1], colors=base_gdf.Color[base], linewidths=2, linestyles='-')
    pc[base], = plt.plot(np.nan, np.nan, color=base_gdf.Color[base], linewidth=2)
plt.axis(np.array((x_min_moa, x_max_moa, y_max_moa, y_min_moa)))
plt.clim(range_set[[0, -1]])
moa_gl.boundary.plot(color='w', ax=ax, linewidth=0.1)
moa_cl.boundary.plot(color='w', ax=ax, linewidth=0.1)
moa_il.boundary.plot(color='w', ax=ax, linewidth=0.1)
ace_gpd.plot(color='g', ax=ax, marker='.', markersize=0.05, label='ACE-SPACE cruise')
plt.title(ac.Name + ' on-station survey range from three viable bases', font='Arial', size=6, fontweight='bold')
plt.xticks(ticks=np.arange(-3e6, 2.5e6 + 5e5, 5e5), labels=np.arange(-3e3, 2.5e3 + 5e2, 5e2).astype(int).astype(str), font='Arial', size=6, fontweight='bold')
plt.yticks(ticks=np.arange(-2.5e6, 2.0e6 + 5e5, 5e5), labels=np.arange(-2.5e3, 2.0e3 + 5e2, 5e2).astype(int).astype(str), font='Arial', size=6, fontweight='bold')
plt.xlabel('EPSG:3031 X (km)', font='Arial', size=6, fontweight='bold')
plt.ylabel('EPSG:3031 Y (km)', font='Arial', size=6, fontweight='bold')
plt.gca().axes.label_outer()
# plt.tick_params(bottom=False, left=False)
plt.grid(linewidth=0.1)
cb_label = range_set.astype(int).astype(str)
cb_label[0] = '≤ ' + cb_label[0]
cb_label[-1] = '≥ ' + cb_label[-1]
cb = plt.colorbar(ims[0], shrink=0.63)
cb.set_ticks(range_set, labels=cb_label, font='Arial', fontsize=6)
cb.set_label('(km)', rotation=270, fontweight='bold')
plt.legend(pc, base_gdf.Name, title='ETOPS limit from:', prop={'family':'Arial', 'size':6, 'weight':'bold'}, borderpad=0.5, loc='lower left', framealpha=0.9)
# plt.legend(pace, prop={'family':'Arial', 'size':6, 'weight':'bold'}, borderpad=0.5, loc='top right', framealpha=0.9)
plt.show()

### 8b) Plot range-to-target grids (Arctic)

In [ ]:
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size']   = 6
plt.rcParams['font.weight'] = 'bold'

num_color = 8
range_set = np.linspace(1.5e3, 5.5e3, 1 + num_color)
cmap = plt.colormaps['cool'].resampled(num_color)
fig = plt.figure(figsize=(4, 4), dpi=300, facecolor='w', layout='constrained')
ax = plt.subplot(1, 1, 1)
plt.imshow(ibcao[::10, ::10], 
           extent=(x_min_ibcao, x_max_ibcao, y_max_ibcao, y_min_ibcao), 
           aspect='equal', 
           cmap='gray')
pc = [None] * num_base
for base in range(num_base):
    alpha_curr = np.full(range_grd.shape[0:2], 0.5)
    alpha_curr[dist_grd_min > ac.RangeETOPS] = 0
    # plt.contour(x_decim, y_decim, dist_ETOPS[:, :, base], levels=[0, 1], colors=base_gdf.Color[base], linewidths=2, linestyles='-')
    pc[base], = plt.plot(np.nan, np.nan, color=base_gdf.Color[base], linewidth=2)
ims = plt.imshow(range_grd_max / 1e3, 
                extent=(np.min(x_ibcao), np.max(x_ibcao), np.min(y_ibcao), np.max(y_ibcao)), 
                aspect='equal', 
                cmap=colors.ListedColormap(cmap(range(num_color))), 
                vmin=range_set[0],
                vmax=range_set[-1],
                alpha=alpha_curr)
plt.axis(np.array((x_min_ibcao, x_max_ibcao, y_max_ibcao, y_min_ibcao)))
plt.clim(range_set[[0, -1]])
plt.title(ac.Name + ' on-station survey range from three viable bases', font='Arial', size=6, fontweight='bold')
# plt.xticks(ticks=np.arange(-3e6, 2.5e6 + 5e5, 5e5), labels=np.arange(-3e3, 2.5e3 + 5e2, 5e2).astype(int).astype(str), font='Arial', size=6, fontweight='bold')
# plt.yticks(ticks=np.arange(-2.5e6, 2.0e6 + 5e5, 5e5), labels=np.arange(-2.5e3, 2.0e3 + 5e2, 5e2).astype(int).astype(str), font='Arial', size=6, fontweight='bold')
plt.xlabel('EPSG:3996 X (km)', font='Arial', size=6, fontweight='bold')
plt.ylabel('EPSG:3996 Y (km)', font='Arial', size=6, fontweight='bold')
plt.gca().axes.label_outer()
# plt.tick_params(bottom=False, left=False)
plt.grid(linewidth=0.1)
cb_label = range_set.astype(int).astype(str)
cb_label[0] = '≤ ' + cb_label[0]
cb_label[-1] = '≥ ' + cb_label[-1]
cb = plt.colorbar(ims, shrink=0.63)
cb.set_ticks(range_set, labels=cb_label, font='Arial', fontsize=6)
cb.set_label('(km)', rotation=270, fontweight='bold')
plt.legend(pc, base_gdf.Name, title='ETOPS limit from:', prop={'family':'Arial', 'size':6, 'weight':'bold'}, borderpad=0.5, loc='lower left', framealpha=0.9)
# plt.legend(pace, prop={'family':'Arial', 'size':6, 'weight':'bold'}, borderpad=0.5, loc='top right', framealpha=0.9)
plt.show()


In [ ]:
plt.figure()
plt.imshow(range_grd_max / 1e3)
plt.colorbar()